In [1]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

/Users/triptibhardwaj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/triptibhardwaj/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [12]:
multiply.invoke({'a': 3, 'b': 5})

15

In [13]:
multiply

StructuredTool(name='multiply', description='Given 2 numbers a and b this tool returns their product', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x129763160>)

Tool Binding

In [24]:
llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

In [25]:
llm.invoke('Hi')

AIMessage(content="Hello! How can I assist you today? Whether it's providing information on various topics or helping with specific tasks, feel free to ask and I'll do my best to help.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-06-28T11:26:03.550512Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11067739959, 'load_duration': 8017806834, 'prompt_eval_count': 30, 'prompt_eval_duration': 251592000, 'eval_count': 37, 'eval_duration': 2761732000, 'logprobs': None, 'model_name': 'qwen2.5:3b'}, id='run--019f0dfa-8fd6-7b30-a446-0d3e03cb42d1-0', usage_metadata={'input_tokens': 30, 'output_tokens': 37, 'total_tokens': 67})

In [26]:
llm_with_tools = llm.bind_tools([multiply])

In [27]:
llm_with_tools.invoke('Hello! How are you?')

AIMessage(content="Hello! I'm just a program, so I don't have feelings. How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-06-28T11:26:12.140366Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4770787375, 'load_duration': 328830666, 'prompt_eval_count': 162, 'prompt_eval_duration': 2006301000, 'eval_count': 23, 'eval_duration': 2372621000, 'logprobs': None, 'model_name': 'qwen2.5:3b'}, id='run--019f0dfa-c9e7-7792-bb86-54bcb21275ec-0', usage_metadata={'input_tokens': 162, 'output_tokens': 23, 'total_tokens': 185})

In [28]:
llm_with_tools.invoke('Can you multiply 3 with 20?').tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 20},
 'id': 'b1a519d0-2011-4cbc-b0b8-724baa5eb202',
 'type': 'tool_call'}

In [29]:
query = HumanMessage('Can you multiply 3 with 20?')

In [30]:
messages = [query]

In [31]:
result = llm_with_tools.invoke(messages)

In [32]:
messages.append(result)

In [33]:
messages

[HumanMessage(content='Can you multiply 3 with 20?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-06-28T11:26:21.071037Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3944437583, 'load_duration': 294558708, 'prompt_eval_count': 166, 'prompt_eval_duration': 43402000, 'eval_count': 38, 'eval_duration': 3600040000, 'logprobs': None, 'model_name': 'qwen2.5:3b'}, id='run--019f0dfa-f01a-7ac0-ba75-52c2106db8a0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 20}, 'id': 'e384514f-9ebe-4925-9725-3f29456583f3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 166, 'output_tokens': 38, 'total_tokens': 204})]

In [34]:
tool_result = multiply.invoke(result.tool_calls[0])

In [35]:
tool_result

ToolMessage(content='60', name='multiply', tool_call_id='e384514f-9ebe-4925-9725-3f29456583f3')

In [36]:
messages.append(tool_result)

In [37]:
messages

[HumanMessage(content='Can you multiply 3 with 20?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-06-28T11:26:21.071037Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3944437583, 'load_duration': 294558708, 'prompt_eval_count': 166, 'prompt_eval_duration': 43402000, 'eval_count': 38, 'eval_duration': 3600040000, 'logprobs': None, 'model_name': 'qwen2.5:3b'}, id='run--019f0dfa-f01a-7ac0-ba75-52c2106db8a0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 20}, 'id': 'e384514f-9ebe-4925-9725-3f29456583f3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 166, 'output_tokens': 38, 'total_tokens': 204}),
 ToolMessage(content='60', name='multiply', tool_call_id='e384514f-9ebe-4925-9725-3f29456583f3')]

In [38]:
llm_with_tools.invoke(messages).content

'The product of 3 and 20 is 60.'